In [1]:
!pip install -q xgboost shap scikit-learn pandas numpy matplotlib seaborn joblib

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

from xgboost import XGBClassifier

import joblib

In [8]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
!find /content/drive/MyDrive -name "Tamil_Nadu_Final_Hazard_Features.csv"

/content/drive/MyDrive/ClimateData/Tamil_Nadu_Final_Hazard_Features.csv


In [10]:
FILE = "/content/drive/MyDrive/ClimateData/Tamil_Nadu_Final_Hazard_Features.csv"

df = pd.read_csv(FILE)

print("Shape:", df.shape)
print(df.head())

Shape: (228950, 25)
         date  district  rainfall  rainfall_1d  rainfall_3d  rainfall_7d  \
0  2010-01-01  Ariyalur      0.00         0.00          NaN          NaN   
1  2010-01-02  Ariyalur      0.22         0.22          NaN          NaN   
2  2010-01-03  Ariyalur      0.26         0.26         0.48          NaN   
3  2010-01-04  Ariyalur      0.00         0.00         0.48          NaN   
4  2010-01-05  Ariyalur      0.00         0.00         0.26          NaN   

   rainfall_30d  rainfall_anomaly  temp_max  temp_min  ...  SPI_3  SPI_6  \
0           NaN         -0.343577     28.27     20.91  ...    NaN    NaN   
1           NaN         -0.393842     27.57     21.17  ...    NaN    NaN   
2           NaN         -0.241747     27.04     21.06  ...    NaN    NaN   
3           NaN         -0.321624     27.01     19.39  ...    NaN    NaN   
4           NaN         -0.372571     27.05     17.39  ...    NaN    NaN   

   temp_anomaly  3_day_temp_mean  7_day_temp_mean  hot_day  hot_da

In [11]:
print("Districts:", df["district"].nunique())

print(
    sorted(df["district"].unique())
)

print(
    "Date range:",
    df["date"].min(),
    "→",
    df["date"].max()
)

print("\nTargets:")
print("Flood:", df["flood"].sum())
print("Heatwave:", df["heatwave"].sum())
print("Drought:", df["drought"].sum())

Districts: 38
['Ariyalur', 'Chengalpattu', 'Chennai', 'Coimbatore', 'Cuddalore', 'Dharmapuri', 'Dindigul', 'Erode', 'Kallakurichi', 'Kancheepuram', 'Kanniyakumari', 'Karur', 'Krishnagiri', 'Madurai', 'Mayiladuthurai', 'Nagapattinam', 'Namakkal', 'Nilgiris', 'Perambalur', 'Pudukkottai', 'Ramanathapuram', 'Ranipet', 'Salem', 'Sivaganga', 'Tenkasi', 'Thanjavur', 'Theni', 'Thoothukudi', 'Tiruchirappalli', 'Tirunelveli', 'Tirupathur', 'Tiruppur', 'Tiruvallur', 'Tiruvannamalai', 'Tiruvarur', 'Vellore', 'Viluppuram', 'Virudhunagar']
Date range: 2010-01-01 → 2026-06-30

Targets:
Flood: 967
Heatwave: 1104
Drought: 68889


In [12]:
df["date"] = pd.to_datetime(df["date"])

df = df.sort_values(
    ["date", "district"]
).reset_index(drop=True)

In [13]:
features = [
    "temp_max",
    "temp_min",
    "temp_mean",
    "temp_range",

    "humidity",
    "wind_speed",
    "pressure",
    "rainfall",
    "soil_wetness",

    "rainfall_3d",
    "rainfall_7d",
    "rainfall_30d",

    "temp_3d_mean",
    "temp_7d_mean",

    "temp_anomaly",
    "rainfall_anomaly",

    "SPI_3",
    "SPI_6",

    "month",
    "day_of_year"
]

In [14]:
df = pd.get_dummies(
    df,
    columns=["district"],
    dtype=int
)

In [15]:
train = df[
    df["date"] < "2022-01-01"
].copy()

validation = df[
    (df["date"] >= "2022-01-01") &
    (df["date"] < "2024-01-01")
].copy()

test = df[
    df["date"] >= "2024-01-01"
].copy()

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (166554, 62)
Validation: (27740, 62)
Test: (34656, 62)


In [16]:
target_columns = [
    "flood",
    "heatwave",
    "drought"
]

In [17]:
feature_columns = [
    col for col in df.columns
    if col not in [
        "date",
        "flood",
        "heatwave",
        "drought"
    ]
]

In [18]:
X_train = train[feature_columns]
X_val = validation[feature_columns]
X_test = test[feature_columns]

y_train = train[target_columns]
y_val = validation[target_columns]
y_test = test[target_columns]

In [19]:
print(
    X_train.isna().sum()
    [lambda x: x > 0]
)

rainfall_3d           76
rainfall_7d          228
rainfall_30d        1102
rainfall_anomaly     141
SPI_3               2242
SPI_6               5738
3_day_temp_mean       76
7_day_temp_mean      228
dtype: int64


In [22]:
import pandas as pd

FILE = "/content/drive/MyDrive/ClimateData/Tamil_Nadu_Final_Hazard_Features.csv"

df = pd.read_csv(FILE)

df["date"] = pd.to_datetime(df["date"])

print(df.columns.tolist())
print("District column exists:", "district" in df.columns)

['date', 'district', 'rainfall', 'rainfall_1d', 'rainfall_3d', 'rainfall_7d', 'rainfall_30d', 'rainfall_anomaly', 'temp_max', 'temp_min', 'temp_mean', 'temp_range', 'humidity', 'wind_speed', 'soil_wetness', 'SPI_3', 'SPI_6', 'temp_anomaly', '3_day_temp_mean', '7_day_temp_mean', 'hot_day', 'hot_day_streak', 'flood', 'heatwave', 'drought']
District column exists: True


In [27]:
missing_rain_anomaly = df[
    df["rainfall_anomaly"].isna()
][
    ["date", "district", "rainfall", "rainfall_anomaly"]
]

print(missing_rain_anomaly.head(20))
print("\nTotal missing:", len(missing_rain_anomaly))
print(
    missing_rain_anomaly["district"].value_counts()
)
missing_rain = df[
    df["rainfall_anomaly"].isna()
][
    ["date", "district", "rainfall", "rainfall_anomaly"]
]

print(missing_rain.head(20))
print("\nTotal missing:", len(missing_rain))

print("\nMissing by district:")
print(missing_rain["district"].value_counts())
print(
    df.loc[
        df["rainfall_anomaly"].isna(),
        "rainfall"
    ].isna().value_counts()
)


            date    district  rainfall  rainfall_anomaly
30182 2010-02-27  Dharmapuri       0.0               NaN
30547 2011-02-27  Dharmapuri       0.0               NaN
30912 2012-02-27  Dharmapuri       0.0               NaN
30914 2012-02-29  Dharmapuri       0.0               NaN
31278 2013-02-27  Dharmapuri       0.0               NaN
31643 2014-02-27  Dharmapuri       0.0               NaN
32008 2015-02-27  Dharmapuri       0.0               NaN
32373 2016-02-27  Dharmapuri       0.0               NaN
32375 2016-02-29  Dharmapuri       0.0               NaN
32739 2017-02-27  Dharmapuri       0.0               NaN
33104 2018-02-27  Dharmapuri       0.0               NaN
33469 2019-02-27  Dharmapuri       0.0               NaN
33834 2020-02-27  Dharmapuri       0.0               NaN
33836 2020-02-29  Dharmapuri       0.0               NaN
34200 2021-02-27  Dharmapuri       0.0               NaN
34565 2022-02-27  Dharmapuri       0.0               NaN
34930 2023-02-27  Dharmapuri   

In [28]:
df["rainfall_anomaly"] = df["rainfall_anomaly"].fillna(0)

In [29]:
print(
    "Missing rainfall anomaly:",
    df["rainfall_anomaly"].isna().sum()
)

Missing rainfall anomaly: 0


In [31]:
print(df.columns.tolist())

['date', 'district', 'rainfall', 'rainfall_1d', 'rainfall_3d', 'rainfall_7d', 'rainfall_30d', 'rainfall_anomaly', 'temp_max', 'temp_min', 'temp_mean', 'temp_range', 'humidity', 'wind_speed', 'soil_wetness', 'SPI_3', 'SPI_6', 'temp_anomaly', '3_day_temp_mean', '7_day_temp_mean', 'hot_day', 'hot_day_streak', 'flood', 'heatwave', 'drought']


In [33]:
ml_features = [
    "temp_max",
    "temp_min",
    "temp_mean",
    "temp_range",
    "humidity",
    "wind_speed",
    "rainfall",
    "soil_wetness",
    "rainfall_3d",
    "rainfall_7d",
    "rainfall_30d",
    "temp_anomaly",
    "rainfall_anomaly",
    "SPI_3",
    "SPI_6"
]

# Keep only features that exist in your CSV
available_features = [
    col for col in ml_features
    if col in df.columns
]

missing_features = [
    col for col in ml_features
    if col not in df.columns
]

print("Available features:")
print(available_features)

print("\nMissing from dataset:")
print(missing_features)
print(
    df[available_features]
    .isna()
    .sum()
    .loc[lambda x: x > 0]
)

Available features:
['temp_max', 'temp_min', 'temp_mean', 'temp_range', 'humidity', 'wind_speed', 'rainfall', 'soil_wetness', 'rainfall_3d', 'rainfall_7d', 'rainfall_30d', 'temp_anomaly', 'rainfall_anomaly', 'SPI_3', 'SPI_6']

Missing from dataset:
[]
rainfall_3d       76
rainfall_7d      228
rainfall_30d    1102
SPI_3           2242
SPI_6           5738
dtype: int64


In [34]:
df["rainfall_anomaly"] = df["rainfall_anomaly"].fillna(0)

print("Missing rainfall anomaly:",
      df["rainfall_anomaly"].isna().sum())

Missing rainfall anomaly: 0


In [35]:
ml_features = [
    "temp_max",
    "temp_min",
    "temp_mean",
    "temp_range",
    "humidity",
    "wind_speed",
    "rainfall",
    "soil_wetness",
    "rainfall_3d",
    "rainfall_7d",
    "rainfall_30d",
    "temp_anomaly",
    "rainfall_anomaly",
    "SPI_3",
    "SPI_6"
]

targets = [
    "flood",
    "heatwave",
    "drought"
]

df_ml = df.dropna(
    subset=ml_features
).copy()

print("Original dataset:", df.shape)
print("ML dataset:", df_ml.shape)
print("Rows removed:", len(df) - len(df_ml))

Original dataset: (228950, 25)
ML dataset: (223212, 25)
Rows removed: 5738


In [36]:
print(
    df_ml[ml_features].isna().sum()
)

temp_max            0
temp_min            0
temp_mean           0
temp_range          0
humidity            0
wind_speed          0
rainfall            0
soil_wetness        0
rainfall_3d         0
rainfall_7d         0
rainfall_30d        0
temp_anomaly        0
rainfall_anomaly    0
SPI_3               0
SPI_6               0
dtype: int64


In [37]:
# ============================================================
# 1. ML FEATURES AND TARGETS
# ============================================================

ml_features = [
    "temp_max",
    "temp_min",
    "temp_mean",
    "temp_range",
    "humidity",
    "wind_speed",
    "rainfall",
    "soil_wetness",
    "rainfall_3d",
    "rainfall_7d",
    "rainfall_30d",
    "temp_anomaly",
    "rainfall_anomaly",
    "SPI_3",
    "SPI_6"
]

targets = [
    "flood",
    "heatwave",
    "drought"
]

# Make sure date is datetime
df["date"] = pd.to_datetime(df["date"])

# Fix the 199 zero-variance rainfall anomaly cases
df["rainfall_anomaly"] = df["rainfall_anomaly"].fillna(0)

# Remove rows where required features are unavailable
df_ml = df.dropna(
    subset=ml_features + targets
).copy()

# Sort chronologically
df_ml = df_ml.sort_values(
    ["date", "district"]
).reset_index(drop=True)

print("Original dataset:", df.shape)
print("ML-ready dataset:", df_ml.shape)
print("Rows removed:", len(df) - len(df_ml))

Original dataset: (228950, 25)
ML-ready dataset: (223212, 25)
Rows removed: 5738


In [38]:
print("\nMissing values in ML features:")

print(
    df_ml[ml_features]
    .isna()
    .sum()
)

print("\nMissing target values:")

print(
    df_ml[targets]
    .isna()
    .sum()
)


Missing values in ML features:
temp_max            0
temp_min            0
temp_mean           0
temp_range          0
humidity            0
wind_speed          0
rainfall            0
soil_wetness        0
rainfall_3d         0
rainfall_7d         0
rainfall_30d        0
temp_anomaly        0
rainfall_anomaly    0
SPI_3               0
SPI_6               0
dtype: int64

Missing target values:
flood       0
heatwave    0
drought     0
dtype: int64


In [39]:
print("\nTarget distribution:")

for target in targets:

    print(f"\n{target.upper()}")

    print(
        df_ml[target]
        .value_counts()
    )

    print(
        "Positive rate:",
        round(
            df_ml[target].mean() * 100,
            3
        ),
        "%"
    )


Target distribution:

FLOOD
flood
0    222267
1       945
Name: count, dtype: int64
Positive rate: 0.423 %

HEATWAVE
heatwave
0    222138
1      1074
Name: count, dtype: int64
Positive rate: 0.481 %

DROUGHT
drought
0    156551
1     66661
Name: count, dtype: int64
Positive rate: 29.864 %


In [40]:
# ============================================================
# 2. TIME-BASED SPLIT
# ============================================================

train = df_ml[
    df_ml["date"] < "2022-01-01"
].copy()

validation = df_ml[
    (df_ml["date"] >= "2022-01-01") &
    (df_ml["date"] < "2024-01-01")
].copy()

test = df_ml[
    df_ml["date"] >= "2024-01-01"
].copy()

print("TRAIN:")
print(train.shape)
print(train["date"].min(), "→", train["date"].max())

print("\nVALIDATION:")
print(validation.shape)
print(validation["date"].min(), "→", validation["date"].max())

print("\nTEST:")
print(test.shape)
print(test["date"].min(), "→", test["date"].max())

TRAIN:
(160816, 25)
2010-06-01 00:00:00 → 2021-12-31 00:00:00

VALIDATION:
(27740, 25)
2022-01-01 00:00:00 → 2023-12-31 00:00:00

TEST:
(34656, 25)
2024-01-01 00:00:00 → 2026-06-30 00:00:00


In [41]:
for name, data in [
    ("TRAIN", train),
    ("VALIDATION", validation),
    ("TEST", test)
]:

    print("\n==============================")
    print(name)
    print("==============================")

    for target in targets:

        positive = data[target].sum()
        total = len(data)

        print(
            f"{target:10s} "
            f"positive={positive:5d} "
            f"rate={positive/total*100:.3f}%"
        )


TRAIN
flood      positive=  941 rate=0.585%
heatwave   positive=  807 rate=0.502%
drought    positive=55683 rate=34.625%

VALIDATION
flood      positive=    4 rate=0.014%
heatwave   positive=   53 rate=0.191%
drought    positive= 3466 rate=12.495%

TEST
flood      positive=    0 rate=0.000%
heatwave   positive=  214 rate=0.617%
drought    positive= 7512 rate=21.676%


In [42]:
X_train = train[
    ml_features + ["district"]
].copy()

X_val = validation[
    ml_features + ["district"]
].copy()

X_test = test[
    ml_features + ["district"]
].copy()

y_train = train[
    targets
].copy()

y_val = validation[
    targets
].copy()

y_test = test[
    targets
].copy()

In [43]:
# ============================================================
# DISTRICT ONE-HOT ENCODING
# ============================================================

X_train = pd.get_dummies(
    X_train,
    columns=["district"],
    dtype=int
)

X_val = pd.get_dummies(
    X_val,
    columns=["district"],
    dtype=int
)

X_test = pd.get_dummies(
    X_test,
    columns=["district"],
    dtype=int
)

# Make validation/test columns exactly match training columns
X_val = X_val.reindex(
    columns=X_train.columns,
    fill_value=0
)

X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (160816, 53)
X_val: (27740, 53)
X_test: (34656, 53)


In [44]:
print("\nFinal feature columns:")
print(X_train.columns.tolist())

print("\nShapes:")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)


Final feature columns:
['temp_max', 'temp_min', 'temp_mean', 'temp_range', 'humidity', 'wind_speed', 'rainfall', 'soil_wetness', 'rainfall_3d', 'rainfall_7d', 'rainfall_30d', 'temp_anomaly', 'rainfall_anomaly', 'SPI_3', 'SPI_6', 'district_Ariyalur', 'district_Chengalpattu', 'district_Chennai', 'district_Coimbatore', 'district_Cuddalore', 'district_Dharmapuri', 'district_Dindigul', 'district_Erode', 'district_Kallakurichi', 'district_Kancheepuram', 'district_Kanniyakumari', 'district_Karur', 'district_Krishnagiri', 'district_Madurai', 'district_Mayiladuthurai', 'district_Nagapattinam', 'district_Namakkal', 'district_Nilgiris', 'district_Perambalur', 'district_Pudukkottai', 'district_Ramanathapuram', 'district_Ranipet', 'district_Salem', 'district_Sivaganga', 'district_Tenkasi', 'district_Thanjavur', 'district_Theni', 'district_Thoothukudi', 'district_Tiruchirappalli', 'district_Tirunelveli', 'district_Tirupathur', 'district_Tiruppur', 'district_Tiruvallur', 'district_Tiruvannamalai', '

In [45]:
print(
    "X_train NaNs:",
    X_train.isna().sum().sum()
)

print(
    "X_val NaNs:",
    X_val.isna().sum().sum()
)

print(
    "X_test NaNs:",
    X_test.isna().sum().sum()
)

X_train NaNs: 0
X_val NaNs: 0
X_test NaNs: 0


In [46]:
print(
    "y_train NaNs:",
    y_train.isna().sum().sum()
)

print(
    "y_val NaNs:",
    y_val.isna().sum().sum()
)

print(
    "y_test NaNs:",
    y_test.isna().sum().sum()
)

y_train NaNs: 0
y_val NaNs: 0
y_test NaNs: 0


In [47]:
# ============================================================
# CLASS IMBALANCE WEIGHTS
# ============================================================

def calculate_scale_pos_weight(y):

    positive = y.sum()
    negative = len(y) - positive

    return negative / positive


flood_weight = calculate_scale_pos_weight(
    y_train["flood"]
)

heatwave_weight = calculate_scale_pos_weight(
    y_train["heatwave"]
)

drought_weight = calculate_scale_pos_weight(
    y_train["drought"]
)

print("Flood weight:", flood_weight)
print("Heatwave weight:", heatwave_weight)
print("Drought weight:", drought_weight)

Flood weight: 169.8990435706695
Heatwave weight: 198.27633209417596
Drought weight: 1.8880627839735646


In [48]:
train_processed = X_train.copy()

for target in targets:
    train_processed[target] = y_train[target].values

val_processed = X_val.copy()

for target in targets:
    val_processed[target] = y_val[target].values

test_processed = X_test.copy()

for target in targets:
    test_processed[target] = y_test[target].values

train_processed.to_csv(
    "/content/train_risk_dataset.csv",
    index=False
)

val_processed.to_csv(
    "/content/validation_risk_dataset.csv",
    index=False
)

test_processed.to_csv(
    "/content/test_risk_dataset.csv",
    index=False
)

print("Saved:")
print("train_risk_dataset.csv")
print("validation_risk_dataset.csv")
print("test_risk_dataset.csv")

Saved:
train_risk_dataset.csv
validation_risk_dataset.csv
test_risk_dataset.csv


In [49]:
from xgboost import XGBClassifier

def create_xgb(scale_pos_weight):
    return XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="aucpr",
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1
    )

In [50]:
flood_model = create_xgb(flood_weight)

flood_model.fit(
    X_train,
    y_train["flood"],
    eval_set=[
        (X_val, y_val["flood"])
    ],
    verbose=False
)

print("Flood model trained.")

Flood model trained.


In [51]:
heatwave_model = create_xgb(heatwave_weight)

heatwave_model.fit(
    X_train,
    y_train["heatwave"],
    eval_set=[
        (X_val, y_val["heatwave"])
    ],
    verbose=False
)

print("Heatwave model trained.")

Heatwave model trained.


In [52]:
drought_model = create_xgb(drought_weight)

drought_model.fit(
    X_train,
    y_train["drought"],
    eval_set=[
        (X_val, y_val["drought"])
    ],
    verbose=False
)

print("Drought model trained.")

Drought model trained.


In [53]:
val_predictions = pd.DataFrame({
    "flood": flood_model.predict_proba(X_val)[:, 1],
    "heatwave": heatwave_model.predict_proba(X_val)[:, 1],
    "drought": drought_model.predict_proba(X_val)[:, 1]
})

val_predictions.head()

,flood,heatwave,drought
0,0.005634,2.347869e-07,0.000001
1,0.000656,2.672487e-07,0.000001
2,0.060118,9.358415e-07,0.000001
3,0.007150,1.856310e-07,0.000001
4,0.003356,4.008163e-07,0.000001


In [54]:
print("VALIDATION TARGET COUNTS")
print("========================")

for target in ["flood", "heatwave", "drought"]:
    print(
        target,
        "positive =",
        y_val[target].sum(),
        "total =",
        len(y_val),
        "rate =",
        round(y_val[target].mean() * 100, 4),
        "%"
    )

VALIDATION TARGET COUNTS
flood positive = 4 total = 27740 rate = 0.0144 %
heatwave positive = 53 total = 27740 rate = 0.1911 %
drought positive = 3466 total = 27740 rate = 12.4946 %


In [55]:
for target in ["flood", "heatwave", "drought"]:

    p = val_predictions[target]

    print("\n", target.upper())

    print("Minimum :", p.min())
    print("Maximum :", p.max())
    print("Mean    :", p.mean())
    print("Median  :", p.median())

    print(
        "Above 0.5:",
        (p >= 0.5).sum()
    )

    print(
        "Above 0.1:",
        (p >= 0.1).sum()
    )


 FLOOD
Minimum : 9.709834785098792e-07
Maximum : 0.9660835862159729
Mean    : 0.012739674
Median  : 0.00104849087074399
Above 0.5: 84
Above 0.1: 706

 HEATWAVE
Minimum : 5.659031465654607e-09
Maximum : 0.9999991655349731
Mean    : 0.0020703988
Median  : 3.322505790492869e-07
Above 0.5: 56
Above 0.1: 64

 DROUGHT
Minimum : 5.962999694020255e-07
Maximum : 0.9999963045120239
Mean    : 0.11997754
Median  : 2.164049419661751e-06
Above 0.5: 3317
Above 0.1: 3363


In [56]:
for target in ["flood", "heatwave", "drought"]:

    events = y_val[target] == 1

    print("\n==============================")
    print(target.upper())
    print("==============================")

    print("Actual events:", events.sum())

    print("\nProbability for actual events:")
    print(
        val_predictions.loc[
            events, target
        ].describe()
    )

    print("\nProbability for non-events:")
    print(
        val_predictions.loc[
            ~events, target
        ].describe()
    )


FLOOD
Actual events: 4

Probability for actual events:


AssertionError: 

In [57]:
# Reset indexes so X/y/predictions all align
X_val = X_val.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)

X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [58]:
val_predictions = pd.DataFrame({
    "flood": flood_model.predict_proba(X_val)[:, 1],
    "heatwave": heatwave_model.predict_proba(X_val)[:, 1],
    "drought": drought_model.predict_proba(X_val)[:, 1]
})

print(val_predictions.head())
print(val_predictions.shape)
print(y_val.shape)

      flood      heatwave   drought
0  0.005634  2.347869e-07  0.000001
1  0.000656  2.672487e-07  0.000001
2  0.060118  9.358415e-07  0.000001
3  0.007150  1.856310e-07  0.000001
4  0.003356  4.008163e-07  0.000001
(27740, 3)
(27740, 3)


In [59]:
for target in ["flood", "heatwave", "drought"]:

    events = y_val[target].to_numpy() == 1
    probabilities = val_predictions[target].to_numpy()

    print("\n==============================")
    print(target.upper())
    print("==============================")

    print("Actual events:", events.sum())

    print("\nProbability for ACTUAL events:")
    print(
        pd.Series(
            probabilities[events]
        ).describe()
    )

    print("\nProbability for NON-events:")
    print(
        pd.Series(
            probabilities[~events]
        ).describe()
    )


FLOOD
Actual events: 4

Probability for ACTUAL events:
count    4.000000
mean     0.045513
std      0.030763
min      0.005323
25%      0.029933
50%      0.051534
75%      0.067114
max      0.073663
dtype: float64

Probability for NON-events:
count    2.773600e+04
mean     1.273495e-02
std      5.187483e-02
min      9.709835e-07
25%      2.302911e-04
50%      1.047761e-03
75%      5.633131e-03
max      9.660836e-01
dtype: float64

HEATWAVE
Actual events: 53

Probability for ACTUAL events:
count    53.000000
mean      0.994384
std       0.025565
min       0.814044
25%       0.998159
50%       0.999801
75%       0.999989
max       0.999999
dtype: float64

Probability for NON-events:
count    2.768700e+04
mean     1.708559e-04
std      8.908241e-03
min      5.659031e-09
25%      2.195921e-07
50%      3.314773e-07
75%      6.219526e-07
max      9.500617e-01
dtype: float64

DROUGHT
Actual events: 3466

Probability for ACTUAL events:
count    3466.000000
mean        0.951203
std         0.2

In [60]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

for target in ["flood", "heatwave", "drought"]:

    y_true = y_val[target].to_numpy()
    probability = val_predictions[target].to_numpy()

    print("\n==============================")
    print(target.upper())
    print("==============================")

    print(
        "ROC-AUC:",
        round(roc_auc_score(y_true, probability), 4)
    )

    print(
        "PR-AUC:",
        round(average_precision_score(y_true, probability), 4)
    )


FLOOD
ROC-AUC: 0.901
PR-AUC: 0.0013

HEATWAVE
ROC-AUC: 1.0
PR-AUC: 0.9997

DROUGHT
ROC-AUC: 0.9999
PR-AUC: 0.9992


In [61]:
print("TEST TARGET COUNTS")
print("==================")

for target in ["flood", "heatwave", "drought"]:
    print(
        target,
        "positive =",
        y_test[target].sum(),
        "total =",
        len(y_test),
        "rate =",
        round(y_test[target].mean() * 100, 4),
        "%"
    )

TEST TARGET COUNTS
flood positive = 0 total = 34656 rate = 0.0 %
heatwave positive = 214 total = 34656 rate = 0.6175 %
drought positive = 7512 total = 34656 rate = 21.6759 %


In [63]:
test_predictions = pd.DataFrame({
    "flood": flood_model.predict_proba(X_test)[:, 1],
    "heatwave": heatwave_model.predict_proba(X_test)[:, 1],
    "drought": drought_model.predict_proba(X_test)[:, 1]
})

print(test_predictions.head())
print("Shape:", test_predictions.shape)

      flood      heatwave       drought
0  0.001607  1.908471e-07  1.018926e-06
1  0.000171  1.761287e-07  9.556709e-07
2  0.001516  9.862893e-07  9.813579e-07
3  0.109409  2.141199e-07  1.021554e-06
4  0.000430  1.984174e-07  1.114984e-06
Shape: (34656, 3)


In [64]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

for target in ["heatwave", "drought"]:

    y_true = y_test[target].to_numpy()
    probability = test_predictions[target].to_numpy()

    print("\n==============================")
    print(target.upper())
    print("==============================")

    print(
        "ROC-AUC:",
        round(
            roc_auc_score(
                y_true,
                probability
            ),
            4
        )
    )

    print(
        "PR-AUC:",
        round(
            average_precision_score(
                y_true,
                probability
            ),
            4
        )
    )


HEATWAVE
ROC-AUC: 1.0
PR-AUC: 0.9964

DROUGHT
ROC-AUC: 0.9998
PR-AUC: 0.9993


In [65]:
flood_by_year = (
    df_ml[df_ml["flood"] == 1]
    .assign(
        year=lambda x: x["date"].dt.year
    )
    .groupby("year")
    .size()
)

print(flood_by_year)

year
2010     63
2011     23
2012      6
2013     15
2014     19
2015    311
2016      4
2017    277
2018     10
2019     66
2021    147
2022      4
dtype: int64


In [66]:
# ============================================================
# FLOOD-SPECIFIC TEMPORAL SPLIT
# ============================================================

flood_train = df_ml[
    df_ml["date"] < "2018-01-01"
].copy()

flood_val = df_ml[
    (df_ml["date"] >= "2018-01-01") &
    (df_ml["date"] < "2020-01-01")
].copy()

flood_test = df_ml[
    (df_ml["date"] >= "2020-01-01") &
    (df_ml["date"] < "2023-01-01")
].copy()

print("Flood TRAIN:")
print(flood_train.shape, "Flood events:", flood_train["flood"].sum())

print("\nFlood VALIDATION:")
print(flood_val.shape, "Flood events:", flood_val["flood"].sum())

print("\nFlood TEST:")
print(flood_test.shape, "Flood events:", flood_test["flood"].sum())

Flood TRAIN:
(105298, 25) Flood events: 718

Flood VALIDATION:
(27740, 25) Flood events: 76

Flood TEST:
(41648, 25) Flood events: 151


In [67]:
flood_features = [
    "temp_max",
    "temp_min",
    "temp_mean",
    "temp_range",
    "humidity",
    "wind_speed",
    "rainfall",
    "soil_wetness",
    "rainfall_3d",
    "rainfall_7d",
    "rainfall_30d",
    "temp_anomaly",
    "rainfall_anomaly",
    "SPI_3",
    "SPI_6"
]

In [68]:
X_flood_train = flood_train[
    flood_features + ["district"]
].copy()

X_flood_val = flood_val[
    flood_features + ["district"]
].copy()

X_flood_test = flood_test[
    flood_features + ["district"]
].copy()

y_flood_train = flood_train["flood"].copy()
y_flood_val = flood_val["flood"].copy()
y_flood_test = flood_test["flood"].copy()

In [69]:
X_flood_train = pd.get_dummies(
    X_flood_train,
    columns=["district"],
    dtype=int
)

X_flood_val = pd.get_dummies(
    X_flood_val,
    columns=["district"],
    dtype=int
)

X_flood_test = pd.get_dummies(
    X_flood_test,
    columns=["district"],
    dtype=int
)

# Make columns identical
X_flood_val = X_flood_val.reindex(
    columns=X_flood_train.columns,
    fill_value=0
)

X_flood_test = X_flood_test.reindex(
    columns=X_flood_train.columns,
    fill_value=0
)

print(X_flood_train.shape)
print(X_flood_val.shape)
print(X_flood_test.shape)

(105298, 53)
(27740, 53)
(41648, 53)


In [70]:
flood_pos = y_flood_train.sum()
flood_neg = len(y_flood_train) - flood_pos

flood_weight = flood_neg / flood_pos

print("Positive:", flood_pos)
print("Negative:", flood_neg)
print("scale_pos_weight:", flood_weight)

Positive: 718
Negative: 104580
scale_pos_weight: 145.65459610027855


In [71]:
from xgboost import XGBClassifier

flood_model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=flood_weight,
    random_state=42,
    n_jobs=-1
)

flood_model.fit(
    X_flood_train,
    y_flood_train,
    eval_set=[
        (X_flood_val, y_flood_val)
    ],
    verbose=False
)

print("New Flood model trained.")

New Flood model trained.


In [72]:
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score
)

flood_val_probability = flood_model.predict_proba(
    X_flood_val
)[:, 1]

print("Flood Validation ROC-AUC:",
      roc_auc_score(
          y_flood_val,
          flood_val_probability
      ))

print("Flood Validation PR-AUC:",
      average_precision_score(
          y_flood_val,
          flood_val_probability
      ))


Flood Validation ROC-AUC: 0.404952474810508
Flood Validation PR-AUC: 0.0025680229283127133


In [73]:
flood_val_prediction = (
    flood_val_probability >= 0.5
).astype(int)

print(
    classification_report(
        y_flood_val,
        flood_val_prediction,
        digits=4,
        zero_division=0
    )
)

              precision    recall  f1-score   support

           0     0.9973    0.9961    0.9967     27664
           1     0.0093    0.0132    0.0109        76

    accuracy                         0.9934     27740
   macro avg     0.5033    0.5046    0.5038     27740
weighted avg     0.9946    0.9934    0.9940     27740



In [75]:
flood_test_probability = flood_model.predict_proba(
    X_flood_test
)[:, 1]

print("Flood Test ROC-AUC:",
      roc_auc_score(
          y_flood_test,
          flood_test_probability
      ))

print("Flood Test PR-AUC:",
      average_precision_score(
          y_flood_test,
          flood_test_probability
      ))
flood_test_prediction = (
    flood_test_probability >= 0.5
).astype(int)

print(
    classification_report(
        y_flood_test,
        flood_test_prediction,
        digits=4,
        zero_division=0
    )
)

Flood Test ROC-AUC: 0.9061647638455311
Flood Test PR-AUC: 0.07447566112075452
              precision    recall  f1-score   support

           0     0.9976    0.9890    0.9933     41497
           1     0.1006    0.3377    0.1550       151

    accuracy                         0.9867     41648
   macro avg     0.5491    0.6634    0.5741     41648
weighted avg     0.9943    0.9867    0.9902     41648



In [76]:
# ============================================================
# FLOOD VALIDATION DIAGNOSTIC
# ============================================================

val_prob = flood_model.predict_proba(
    X_flood_val
)[:, 1]

val_events = y_flood_val.to_numpy() == 1

print("VALIDATION")
print("==============================")
print("Actual flood events:", val_events.sum())

print("\nProbability for ACTUAL floods:")
print(
    pd.Series(
        val_prob[val_events]
    ).describe()
)

print("\nProbability for NON-flood observations:")
print(
    pd.Series(
        val_prob[~val_events]
    ).describe()
)

VALIDATION
Actual flood events: 76

Probability for ACTUAL floods:
count    76.000000
mean      0.014014
std       0.087017
min       0.000017
25%       0.000054
50%       0.000237
75%       0.000888
max       0.755033
dtype: float64

Probability for NON-flood observations:
count    2.766400e+04
mean     1.075235e-02
std      5.637647e-02
min      9.420559e-07
25%      1.199793e-04
50%      5.719785e-04
75%      2.983499e-03
max      9.729110e-01
dtype: float64


In [77]:
# Compare the actual climate conditions
# between flood and non-flood observations

print("\nFlood-event feature means:")
print(
    flood_val.loc[
        flood_val["flood"] == 1,
        flood_features
    ].mean()
)

print("\nNon-flood feature means:")
print(
    flood_val.loc[
        flood_val["flood"] == 0,
        flood_features
    ].mean()
)


Flood-event feature means:
temp_max            33.386053
temp_min            19.544342
temp_mean           26.465197
temp_range          13.841711
humidity            59.450921
wind_speed           2.033289
rainfall             1.648684
soil_wetness         0.540000
rainfall_3d          4.719474
rainfall_7d          7.078421
rainfall_30d        15.716974
temp_anomaly         1.507199
rainfall_anomaly     0.065523
SPI_3               -1.395639
SPI_6               -0.215981
dtype: float64

Non-flood feature means:
temp_max            32.926628
temp_min            23.260790
temp_mean           28.093709
temp_range           9.665838
humidity            70.259838
wind_speed           2.722927
rainfall             2.513624
soil_wetness         0.566101
rainfall_3d          7.537136
rainfall_7d         17.586746
rainfall_30d        74.805622
temp_anomaly         0.429415
rainfall_anomaly    -0.073688
SPI_3               -0.161306
SPI_6               -0.314857
dtype: float64


In [78]:
features = [
    "temp_max",
    "temp_min",
    "temp_mean",
    "temp_range",
    "humidity",
    "wind_speed",
    "rainfall",
    "soil_wetness",
    "rainfall_3d",
    "rainfall_7d",
    "rainfall_30d",
    "temp_anomaly",
    "rainfall_anomaly",
    "SPI_3",
    "SPI_6"
]

In [79]:
new_data = flood_test.iloc[[0]].copy()

X_new = new_data[
    flood_features + ["district"]
].copy()

X_new = pd.get_dummies(
    X_new,
    columns=["district"],
    dtype=int
)

X_new = X_new.reindex(
    columns=X_flood_train.columns,
    fill_value=0
)

flood_probability = flood_model.predict_proba(
    X_new
)[:, 1][0]

print("District:", new_data["district"].iloc[0])
print("Flood probability:", flood_probability)

District: Ariyalur
Flood probability: 0.00068231137


In [80]:
new_data = test.iloc[[0]].copy()

X_new = new_data[
    ml_features + ["district"]
].copy()

X_new = pd.get_dummies(
    X_new,
    columns=["district"],
    dtype=int
)

X_new = X_new.reindex(
    columns=X_train.columns,
    fill_value=0
)

flood_probability = flood_model.predict_proba(
    X_new
)[:, 1][0]

heatwave_probability = heatwave_model.predict_proba(
    X_new
)[:, 1][0]

drought_probability = drought_model.predict_proba(
    X_new
)[:, 1][0]

print("District:", new_data["district"].iloc[0])

print("Flood:", flood_probability)
print("Heatwave:", heatwave_probability)
print("Drought:", drought_probability)

District: Ariyalur
Flood: 0.00055459887
Heatwave: 1.9084708e-07
Drought: 1.0189259e-06


In [81]:
def risk_level(p):
    if p >= 0.70:
        return "HIGH"
    elif p >= 0.40:
        return "MEDIUM"
    else:
        return "LOW"
print("Flood:",
      risk_level(flood_probability))

print("Heatwave:",
      risk_level(heatwave_probability))

print("Drought:",
      risk_level(drought_probability))

Flood: LOW
Heatwave: LOW
Drought: LOW
